# 不确定性分析

## Ip

读取四条剖面的 Ip 真实值及 HCT-QHead 的 low、median、upper 预测，绘制相对误差、不确定性和合并分布图，并输出汇总表。该单元同时定义后续 Is、Dn 共用的分析函数。

In [ ]:
# Ip：公共分析函数 + Ip 绘图
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# 与参考 Notebook 一致：使用 Matplotlib 默认字体和字号。
plt.rcdefaults()


def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "HCT-QHead" / "byprofile").is_dir() and (
            candidate / "section"
        ).is_dir():
            return candidate
    raise FileNotFoundError("未找到项目根目录 HCT-QHead_Synthesis_AVO。")


PROJECT_ROOT = find_project_root()
PRED_ROOT = PROJECT_ROOT / "HCT-QHead" / "byprofile"
TRUTH_ROOT = PROJECT_ROOT / "section"
OUTPUT_DIR = PROJECT_ROOT / "analysis" /"uncertainty"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROFILE_NAMES = [f"profile_s{i}" for i in range(1, 5)]
PROFILE_TIME_RANGES = {name: (2.0, 2.8) for name in PROFILE_NAMES}
PREDICTION_LABELS = ("low", "median", "upper")
COLOR_PERCENTILES = (2.0, 98.0)
HISTOGRAM_BINS = 50
CMAP_NAME = "magma_r"
SAVE_DPI = 300
SHOW_FIGURES = True


def clean_columns(dataframe):
    dataframe = dataframe.copy()
    dataframe.columns = [
        str(column).replace("\ufeff", "").strip() for column in dataframe.columns
    ]
    return dataframe


def find_time_column(dataframe):
    for column in dataframe.columns:
        if str(column).lower().startswith("time"):
            return column
    raise ValueError(f"未找到时间列，当前列名：{list(dataframe.columns)}")


def read_matrix(csv_path):
    if not csv_path.exists():
        raise FileNotFoundError(f"文件不存在：{csv_path}")
    dataframe = clean_columns(pd.read_csv(csv_path))
    time_column = find_time_column(dataframe)
    time = pd.to_numeric(dataframe[time_column], errors="coerce").to_numpy(dtype=float)
    data_columns = [column for column in dataframe.columns if column != time_column]
    matrix = (
        dataframe[data_columns]
        .apply(pd.to_numeric, errors="coerce")
        .to_numpy(dtype=float)
    )
    matrix[(matrix == 0.0) | ~np.isfinite(matrix)] = np.nan
    if not np.isfinite(time).all():
        raise ValueError(f"时间列含 NaN 或 Inf：{csv_path}")
    return time, data_columns, matrix


def assert_aligned(reference, candidate, reference_path, candidate_path):
    ref_time, ref_columns, ref_matrix = reference
    time, columns, matrix = candidate
    if ref_columns != columns:
        raise ValueError(f"道列不一致：{reference_path} 与 {candidate_path}")
    if ref_matrix.shape != matrix.shape:
        raise ValueError(f"矩阵形状不一致：{reference_path} 与 {candidate_path}")
    if not np.array_equal(ref_time, time, equal_nan=True):
        raise ValueError(f"时间列不一致：{reference_path} 与 {candidate_path}")


def safe_divide(numerator, denominator):
    result = np.full(numerator.shape, np.nan, dtype=float)
    valid = np.isfinite(numerator) & np.isfinite(denominator) & (denominator != 0.0)
    return np.divide(numerator, denominator, out=result, where=valid)


def load_target(target):
    target_data = {}
    for profile_name in PROFILE_NAMES:
        prediction_dir = PRED_ROOT / profile_name
        paths = {
            label: prediction_dir / f"{target}_{label}.csv"
            for label in PREDICTION_LABELS
        }
        truth_path = TRUTH_ROOT / profile_name / f"{target}.csv"

        median_data = read_matrix(paths["median"])
        low_data = read_matrix(paths["low"])
        upper_data = read_matrix(paths["upper"])
        truth_data = read_matrix(truth_path)
        assert_aligned(median_data, low_data, paths["median"], paths["low"])
        assert_aligned(median_data, upper_data, paths["median"], paths["upper"])
        assert_aligned(median_data, truth_data, paths["median"], truth_path)

        time, _, median = median_data
        low = low_data[2]
        upper = upper_data[2]
        truth = truth_data[2]

        time_range = PROFILE_TIME_RANGES.get(profile_name)
        time_mask = np.ones(time.shape, dtype=bool)
        if time_range is not None:
            time_start, time_end = time_range
            if time_start is not None:
                time_mask &= time >= float(time_start)
            if time_end is not None:
                time_mask &= time <= float(time_end)

        time = time[time_mask]
        median = median[time_mask]
        low = low[time_mask]
        upper = upper[time_mask]
        truth = truth[time_mask]

        valid = (
            np.isfinite(truth)
            & np.isfinite(median)
            & np.isfinite(low)
            & np.isfinite(upper)
        )
        absolute_error = np.where(valid, np.abs(truth - median), np.nan)
        # 相对误差以真实值绝对值归一化；真实值为零时由 safe_divide 返回 NaN。
        relative_error = safe_divide(absolute_error, np.abs(truth))
        # 相对不确定性仍按预测区间宽度 / 预测中位数定义。
        uncertainty = safe_divide(upper - low, median)

        target_data[profile_name] = {
            "time": time,
            "median": median,
            "truth": truth,
            "relative_error": relative_error,
            "uncertainty": uncertainty,
        }
    return target_data


def resolve_cmap_range(target_data, field, scale, manual_range):
    if manual_range is not None:
        if not isinstance(manual_range, (tuple, list)) or len(manual_range) != 2:
            raise ValueError("手动色标必须为 None 或 (vmin, vmax)。")
        vmin, vmax = map(float, manual_range)
        if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin >= vmax:
            raise ValueError(f"手动色标范围无效：{manual_range}")
        return vmin, vmax, "手动"

    parts = []
    for profile_name in PROFILE_NAMES:
        values = target_data[profile_name][field] * scale
        values = values[np.isfinite(values)]
        if values.size:
            parts.append(values)
    if not parts:
        raise ValueError(f"{field} 没有有效值。")

    all_values = np.concatenate(parts)
    vmin, vmax = np.percentile(all_values, COLOR_PERCENTILES).astype(float)
    if vmax <= vmin:
        delta = max(abs(vmin), 1.0) * 1e-6
        vmin, vmax = vmin - delta, vmax + delta
    source = f"{COLOR_PERCENTILES[0]:g}%–{COLOR_PERCENTILES[1]:g}%"
    return vmin, vmax, source


def plot_profile_strip(
    target_data, target, field, scale, manual_range, cbar_label, filename
):
    vmin, vmax, range_source = resolve_cmap_range(
        target_data, field, scale, manual_range
    )
    print(f"{target} {field}: {range_source} 色标范围 = {vmin:.6g} ～ {vmax:.6g}")

    figure = plt.figure(figsize=(6, 6))
    grid = figure.add_gridspec(4, 2, width_ratios=[1, 0.035], hspace=0.22, wspace=0.08)
    axes = [figure.add_subplot(grid[index, 0]) for index in range(4)]
    colorbar_ax = figure.add_subplot(grid[:, 1])
    figure.subplots_adjust(
        left=0.13, right=0.86, bottom=max(0.06, 0.5 / figure.get_figheight()), top=0.98
    )

    cmap = plt.colormaps[CMAP_NAME].copy()
    cmap.set_bad(color="white")
    for index, (profile_name, ax) in enumerate(zip(PROFILE_NAMES, axes), start=1):
        time = target_data[profile_name]["time"]
        values = target_data[profile_name][field] * scale
        image = ax.imshow(
            values,
            cmap=cmap,
            aspect="auto",
            interpolation="none",
            origin="upper",
            extent=[1, values.shape[1], float(time.max()), float(time.min())],
            vmin=vmin,
            vmax=vmax,
        )
        ax.set_ylabel("Time (s)")
        ax.set_ylim(2.8, 2.0)
    axes[-1].set_xlabel("Trace")
    colorbar = figure.colorbar(image, cax=colorbar_ax, extend="neither")
    colorbar.set_label(cbar_label)

    output_path = OUTPUT_DIR / filename
    figure.savefig(output_path, dpi=SAVE_DPI, facecolor="white")
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(figure)
    print(f"已保存：{output_path}")


def collect_uncertainty_samples(target_data, high_error_threshold_pct):
    samples = {}
    threshold = float(high_error_threshold_pct) / 100.0
    for index, profile_name in enumerate(PROFILE_NAMES, start=1):
        uncertainty = target_data[profile_name]["uncertainty"]
        relative_error = target_data[profile_name]["relative_error"]
        valid = np.isfinite(uncertainty) & np.isfinite(relative_error)
        samples[index] = (
            uncertainty[valid],
            uncertainty[valid & (relative_error > threshold)],
        )
    return samples


def build_summary(samples, target, high_error_threshold_pct):
    threshold_text = f"{high_error_threshold_pct:g}%"
    high_median_column = f"Median uncertainty (>{threshold_text} error)"
    rows = []
    for index, (all_values, high_values) in samples.items():
        median_all = np.median(all_values)
        median_high = np.median(high_values) if high_values.size else np.nan
        rows.append(
            {
                "Profile": f"S{index}",
                "High-error samples (%)": high_values.size / all_values.size * 100,
                "Median uncertainty (all)": median_all,
                high_median_column: median_high,
                "Difference": median_high - median_all,
            }
        )

    all_profiles = np.concatenate([values[0] for values in samples.values()])
    high_profiles = np.concatenate([values[1] for values in samples.values()])
    median_all = np.median(all_profiles)
    median_high = np.median(high_profiles) if high_profiles.size else np.nan
    rows.append(
        {
            "Profile": "All",
            "High-error samples (%)": high_profiles.size / all_profiles.size * 100,
            "Median uncertainty (all)": median_all,
            high_median_column: median_high,
            "Difference": median_high - median_all,
        }
    )

    summary = pd.DataFrame(rows)
    output_path = OUTPUT_DIR / f"HCT-QHead_Uncertainty_Summary_{target}.csv"
    summary.to_csv(output_path, index=False)
    print(f"已保存：{output_path}")
    return summary, all_profiles, high_profiles


def plot_total_distribution(
    all_profiles, high_profiles, target, high_error_threshold_pct
):
    bins = np.histogram_bin_edges(all_profiles, bins=HISTOGRAM_BINS)
    threshold_text = f"{high_error_threshold_pct:g}%"
    figure, ax = plt.subplots(figsize=(6, 3))
    ax.hist(
        all_profiles,
        bins=bins,
        weights=np.ones(all_profiles.size) / all_profiles.size,
        color="steelblue",
        edgecolor="white",
        alpha=0.55,
        label="All samples",
    )
    if high_profiles.size:
        ax.hist(
            high_profiles,
            bins=bins,
            weights=np.ones(high_profiles.size) / high_profiles.size,
            color="tomato",
            edgecolor="white",
            alpha=0.55,
            label=f"Relative Error > {threshold_text}",
        )

    median_all = np.median(all_profiles)
    median_high = np.median(high_profiles) if high_profiles.size else np.nan
    ax.axvline(
        median_all,
        color="steelblue",
        linestyle="--",
        linewidth=1.5,
        label=f"Median (All): {median_all:.3f}",
    )
    if np.isfinite(median_high):
        ax.axvline(
            median_high,
            color="tomato",
            linestyle="--",
            linewidth=1.5,
            label=f"Median (>{threshold_text}): {median_high:.3f}",
        )
    ax.set(xlabel="Relative uncertainty", ylabel="Frequency")
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda value, _: f"{value * 100:.0f}%")
    )
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.legend(frameon=False)
    figure.tight_layout()

    output_path = OUTPUT_DIR / f"HCT-QHead_Uncertainty_Distribution_All_{target}.png"
    figure.savefig(output_path, facecolor="white")
    if SHOW_FIGURES:
        plt.show()
    else:
        plt.close(figure)
    print(f"已保存：{output_path}")


def format_summary(summary):
    formats = {
        "High-error samples (%)": "{:.2f}%",
        "Median uncertainty (all)": "{:.4f}",
        "Difference": "{:+.4f}",
    }
    formats.update(
        {
            column: "{:.4f}"
            for column in summary.columns
            if column.startswith("Median uncertainty (>")
        }
    )
    return summary.style.format(formats).hide(axis="index")


def analyze_target(
    target,
    high_error_threshold_pct,
    relative_error_cmap_range=None,
    uncertainty_cmap_range=None,
):
    high_error_threshold_pct = float(high_error_threshold_pct)
    if not np.isfinite(high_error_threshold_pct) or high_error_threshold_pct < 0:
        raise ValueError("高相对误差阈值必须是大于或等于 0 的有限百分数。")
    target_data = load_target(target)
    plot_profile_strip(
        target_data,
        target,
        "relative_error",
        100.0,
        relative_error_cmap_range,
        "Relative Error (%)",
        f"HCT-QHead_RelError_{target}.png",
    )
    plot_profile_strip(
        target_data,
        target,
        "uncertainty",
        1.0,
        uncertainty_cmap_range,
        "Relative uncertainty",
        f"HCT-QHead_Uncertainty_{target}.png",
    )
    samples = collect_uncertainty_samples(
        target_data, high_error_threshold_pct
    )
    summary, all_profiles, high_profiles = build_summary(
        samples, target, high_error_threshold_pct
    )
    plot_total_distribution(
        all_profiles, high_profiles, target, high_error_threshold_pct
    )
    return target_data, summary


# Ip 独立色标接口：None 使用四条剖面合并值的 2%～98%；也可设为 (vmin, vmax)。
IP_REL_ERROR_CMAP_RANGE = (0,10)
IP_UNCERTAINTY_CMAP_RANGE = None
IP_HIGH_ERROR_THRESHOLD_PCT = 10
ip_data, ip_summary = analyze_target(
    "Ip",
    high_error_threshold_pct=IP_HIGH_ERROR_THRESHOLD_PCT,
    relative_error_cmap_range=IP_REL_ERROR_CMAP_RANGE,
    uncertainty_cmap_range=IP_UNCERTAINTY_CMAP_RANGE,
)
display(format_summary(ip_summary))

## Is 分析

使用独立的 Is 色标接口，绘制四条剖面的相对误差、不确定性和合并分布图，并输出汇总表。运行前需先执行上方 Ip 单元以加载公共函数。

In [ ]:
# Is：独立色标接口与绘图
# None 使用四条剖面合并值的 2%～98%；也可设为 (vmin, vmax)。
IS_REL_ERROR_CMAP_RANGE = (0,10)
IS_UNCERTAINTY_CMAP_RANGE = None
IS_HIGH_ERROR_THRESHOLD_PCT = 10

is_data, is_summary = analyze_target(
    "Is",
    high_error_threshold_pct=IS_HIGH_ERROR_THRESHOLD_PCT,
    relative_error_cmap_range=IS_REL_ERROR_CMAP_RANGE,
    uncertainty_cmap_range=IS_UNCERTAINTY_CMAP_RANGE,
)
display(format_summary(is_summary))

## Dn 分析

使用独立的 Dn 色标接口，绘制四条剖面的相对误差、不确定性和合并分布图，并输出汇总表。运行前需先执行上方 Ip 单元以加载公共函数。

In [ ]:
# Dn：独立色标接口与绘图
# None 使用四条剖面合并值的 2%～98%；也可设为 (vmin, vmax)。
DN_REL_ERROR_CMAP_RANGE = (0,3.5)
DN_UNCERTAINTY_CMAP_RANGE = None
DN_HIGH_ERROR_THRESHOLD_PCT = 3.5

dn_data, dn_summary = analyze_target(
    "Dn",
    high_error_threshold_pct=DN_HIGH_ERROR_THRESHOLD_PCT,
    relative_error_cmap_range=DN_REL_ERROR_CMAP_RANGE,
    uncertainty_cmap_range=DN_UNCERTAINTY_CMAP_RANGE,
)
display(format_summary(dn_summary))

# 表格

汇总 Ip、Is 和 Dn 的 `All` 行不确定性统计，并生成可直接粘贴到论文中的 LaTex 表格。运行此单元前请先依次运行上方三个参数分析单元。

In [ ]:
# 三个参数的 All 行汇总表与 LaTex 代码
# 运行前请先运行 Ip、Is、Dn 分析单元，以生成 ip_summary、is_summary 和 dn_summary。
TABLE_SPECS = [
    ("P-wave impedance", ip_summary, IP_HIGH_ERROR_THRESHOLD_PCT),
    ("S-wave impedance", is_summary, IS_HIGH_ERROR_THRESHOLD_PCT),
    ("Density", dn_summary, DN_HIGH_ERROR_THRESHOLD_PCT),
]

table_rows = []
for parameter, summary, threshold_pct in TABLE_SPECS:
    high_median_column = f"Median uncertainty (>{threshold_pct:g}% error)"
    all_row = summary.loc[summary["Profile"].eq("All")].iloc[0]
    table_rows.append(
        {
            "Parameter": parameter,
            "High-error samples (%)": all_row["High-error samples (%)"],
            "Median uncertainty (all)": all_row["Median uncertainty (all)"],
            "Median uncertainty (high-error)": all_row[high_median_column],
            "Difference": all_row["Difference"],
        }
    )

all_summary_table = pd.DataFrame(table_rows).set_index("Parameter")
TABLE_METRICS = [
    ("High error samples (%)", "High-error samples (%)", lambda value: f"{value:.2f}%"),
    ("Median uncertainty (all samples)", "Median uncertainty (all)", lambda value: f"{value:.4f}"),
    ("Median uncertainty (high-error samples)", "Median uncertainty (high-error)", lambda value: f"{value:.4f}"),
    ("Difference", "Difference", lambda value: f"{value:+.4f}"),
]

transposed_table = pd.DataFrame(
    {
        "Statistic": [display_name for display_name, _, _ in TABLE_METRICS],
        **{
            parameter: [
                formatter(all_summary_table.loc[parameter, source_column])
                for _, source_column, formatter in TABLE_METRICS
            ]
            for parameter in all_summary_table.index
        },
    }
)
display(transposed_table.style.hide(axis="index"))

LATEX_BACKSLASH = chr(92)
LATEX_LINEBREAK = LATEX_BACKSLASH * 2
LATEX_PERCENT = LATEX_BACKSLASH + "%"
TEX_METRICS = [
    (f"High error samples ({LATEX_PERCENT})", "High-error samples (%)", lambda value: f"{value:.2f}{LATEX_PERCENT}"),
    (f"{LATEX_BACKSLASH}shortstack[l]{{Median uncertainty {LATEX_LINEBREAK} (all samples)}}", "Median uncertainty (all)", lambda value: f"{value:.4f}"),
    (f"{LATEX_BACKSLASH}shortstack[l]{{Median uncertainty {LATEX_LINEBREAK} (high-error samples)}}", "Median uncertainty (high-error)", lambda value: f"{value:.4f}"),
    ("Difference", "Difference", lambda value: f"{value:+.4f}"),
]
tex_rows = "\n".join(
    f"\t\t\t{label} & "
    + " & ".join(
        formatter(all_summary_table.loc[parameter, source_column])
        for parameter in all_summary_table.index
    )
    + f" {LATEX_LINEBREAK}"
    for label, source_column, formatter in TEX_METRICS
)

latex_table = rf"""\begin{{table}}[htp]
    \centering
    \caption{{Relative uncertainty statistics for all samples of the three elastic parameters.}}
    \label{{tab:qhead_uncertainty_all_profiles}}
    \renewcommand{{\arraystretch}}{{1.10}}
    \setlength{{\tabcolsep}}{{3pt}}
    \resizebox{{\columnwidth}}{{!}}{{
        \begin{{tabular}}{{lccc}}
            \toprule
            Statistic & P-wave impedance & S-wave impedance & Density {LATEX_LINEBREAK}
            \midrule
{tex_rows}
            \bottomrule
        \end{{tabular}}
    }}
\end{{table}}"""

print(latex_table)

# end